# Notebook 04 - Clean Data and Build Features

Goes with **Lesson 06**. We clean `student_performance_messy.csv`, which has the same kind of data as before but with real-world mistakes.

The recipe:

```
load -> look -> fix text -> remove duplicates -> fix impossible values -> fill missing -> encode -> engineer features -> scale -> save
```

In [ ]:
from pathlib import Path

# The data files are in the "data" folder next to "notebooks".
# In Google Colab, upload the CSV file and it will be found in the current folder.
DATA_DIR = Path("../data") if Path("../data").exists() else Path(".")
print("Reading data from:", DATA_DIR.resolve())

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(DATA_DIR / "student_performance_messy.csv")
print("Shape:", df.shape)
df.head()

## 1. Look for problems

In [ ]:
print("Missing values in each column:")
print(df.isna().sum())
print()
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
print(df["internet_access"].value_counts())
print()
print(df["city_type"].value_counts())

See the problem? `yes`, `Yes`, ` YES` and `NO` are the same answers written differently.

In [ ]:
df[["hours_studied", "attendance_percent", "sleep_hours"]].describe().round(1)

Look at the **min** and **max** rows: attendance of 450 percent, 60 hours of study per day and negative sleep are **impossible**. These are typing mistakes.

## 2. Fix messy text

`.str.strip()` removes spaces at the start and end. `.str.lower()` makes everything small letters.

In [ ]:
text_columns = ["internet_access", "extra_classes", "city_type", "result"]
for column in text_columns:
    df[column] = df[column].str.strip().str.lower()

print(df["internet_access"].value_counts())
print()
print(df["city_type"].value_counts())

## 3. Remove duplicates

In [ ]:
before = len(df)
df = df.drop_duplicates()
print("Removed", before - len(df), "duplicate rows. Rows now:", len(df))

## 4. Fix impossible values

We use common-sense rules. A value that breaks the rule becomes "missing" (`NaN`), and we fill it in the next step.

In [ ]:
df.loc[(df["attendance_percent"] < 0) | (df["attendance_percent"] > 100), "attendance_percent"] = np.nan
df.loc[(df["hours_studied"] < 0) | (df["hours_studied"] > 16), "hours_studied"] = np.nan
df.loc[(df["sleep_hours"] < 0) | (df["sleep_hours"] > 16), "sleep_hours"] = np.nan

df[["hours_studied", "attendance_percent", "sleep_hours"]].describe().round(1)

### Extra: the IQR rule for finding outliers

The IQR rule finds values that are unusually far from the middle. Here it finds nothing, because we already removed the mistakes. Real but rare values (like 9.5 hours of study) are fine to keep.

In [ ]:
q1 = df["hours_studied"].quantile(0.25)
q3 = df["hours_studied"].quantile(0.75)
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = df[(df["hours_studied"] < low) | (df["hours_studied"] > high)]
print(f"Normal range for hours_studied: {low:.1f} to {high:.1f}")
print("Outliers found:", len(outliers))

## 5. Fill missing values

Number columns: fill with the **median**. (We would use the **mode** for text columns, but they have no missing values here.)

In [ ]:
for column in ["hours_studied", "attendance_percent", "sleep_hours"]:
    median_value = df[column].median()
    df[column] = df[column].fillna(median_value)
    print(f"Filled {column} with median {median_value}")

print()
print("Missing values now:")
print(df.isna().sum())

## 6. Encoding: turn categories into numbers

Yes/no columns become 1/0. `city_type` has 3 groups, so we use **one-hot encoding**.

In [ ]:
df["internet_access"] = df["internet_access"].map({"yes": 1, "no": 0})
df["extra_classes"] = df["extra_classes"].map({"yes": 1, "no": 0})

df = pd.get_dummies(df, columns=["city_type"], prefix="city", dtype=int)
df.head()

## 7. Feature engineering: create new useful columns

In [ ]:
df["well_rested"] = (df["sleep_hours"] >= 7).astype(int)
df["study_per_sleep"] = (df["hours_studied"] / df["sleep_hours"]).round(2)

df[["hours_studied", "sleep_hours", "well_rested", "study_per_sleep"]].head()

## 8. Feature selection: remove columns that cannot help

`student_id` is only a name tag. It has no information about marks.

In [ ]:
df = df.drop(columns=["student_id"])
print("Columns now:", list(df.columns))

## 9. Scaling (demonstration)

`StandardScaler` makes each column have mean 0 and standard deviation 1. We only show it here. In the next notebooks we scale **inside** the model pipeline, so we save the file **without** scaling.

In [ ]:
from sklearn.preprocessing import StandardScaler

number_columns = ["hours_studied", "attendance_percent", "sleep_hours", "previous_score"]
scaled = StandardScaler().fit_transform(df[number_columns])
scaled_df = pd.DataFrame(scaled, columns=number_columns)

print("Before scaling:")
print(df[number_columns].describe().loc[["mean", "std", "min", "max"]].round(1))
print()
print("After scaling:")
print(scaled_df.describe().loc[["mean", "std", "min", "max"]].round(1))

## 10. Final check and save

In [ ]:
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicates:    ", int(df.duplicated().sum()))
print("Max attendance:", df["attendance_percent"].max())
print("Min sleep:     ", df["sleep_hours"].min())

output_file = DATA_DIR / "student_performance_cleaned.csv"
df.to_csv(output_file, index=False)
print()
print("Saved clean data to", output_file)
df.head()

## Data leakage reminder

If we want to predict `result`, we must **not** use `final_score` as a feature, because `result` is made from `final_score`. The model would "cheat". In the next notebooks we always remove `final_score` when predicting `result`.

Go back to **Lesson 06** and do the **Check** section.